In [1]:
import chromadb
import os

chroma_client = chromadb.Client()

# Crear o recuperar la colección
# ChromaDB usará su función de embedding por defecto para vectorizar los textos
collection = chroma_client.get_or_create_collection(name="seguridad_bd_collection")

print("Cliente inicializado y colección lista.")

Cliente inicializado y colección lista.


In [2]:
# Definir la ruta donde están los archivos .txt
# Usamos una ruta relativa estándar 
carpeta_documentos = os.path.join("documentos")

documentos = []
ids_documentos = []

# Leer cada archivo .txt en la carpeta
for nombre_archivo in os.listdir(carpeta_documentos):
    if nombre_archivo.endswith(".txt"):
        ruta_completa = os.path.join(carpeta_documentos, nombre_archivo)
        
        # Abrir y leer el contenido del archivo
        with open(ruta_completa, "r", encoding="utf-8") as archivo:
            contenido_texto = archivo.read()
            
            # Agregar el contenido y un ID
            documentos.append(contenido_texto)
            ids_documentos.append(nombre_archivo)

print(f"Se encontraron {len(documentos)} documentos listos para indexar.")

Se encontraron 4 documentos listos para indexar.


In [3]:
# Hacer el upsert a ChromaDB
# Aquí ChromaDB convierte los textos a embeddings y los guarda
if documentos:
    collection.upsert(
        documents=documentos,
        ids=ids_documentos
    )
    print(f"Éxito: Se indexaron y vectorizaron {len(documentos)} documentos en ChromaDB.")
else:
    print("No se encontraron documentos para indexar. Revisa la ruta de tu carpeta.")

Éxito: Se indexaron y vectorizaron 4 documentos en ChromaDB.


In [4]:
consulta = "¿Por qué las soluciones NoSQL, como Firebase Firestore, guardan la información en colecciones de documentos flexibles?"

# Consulta de prueba (Texto -> Embedding -> Búsqueda de similitud)
resultados = collection.query(
    query_texts=[consulta],
    n_results=2 
)

# Imprimir los resultados
print(f"Resultados para la consulta: '{consulta}'\n")
print("-" * 50)

# Extraer los datos del diccionario de resultados
ids_recuperados = resultados['ids'][0]
documentos_recuperados = resultados['documents'][0]
distancias = resultados['distances'][0]

for i in range(len(ids_recuperados)):
    print(f"Documento (ID): {ids_recuperados[i]}")
    # La distancia indica qué tan cerca está el embedding de la consulta al documento
    # En ChromaDB, menor distancia (más cerca a 0) significa mayor similitud
    print(f"Distancia/Similitud: {distancias[i]:.4f}") 
    print(f"Contenido:\n{documentos_recuperados[i]}")
    print("-" * 50)

Resultados para la consulta: '¿Por qué las soluciones NoSQL, como Firebase Firestore, guardan la información en colecciones de documentos flexibles?'

--------------------------------------------------
Documento (ID): tipos_de_BD.txt
Distancia/Similitud: 0.6041
Contenido:
Al diseñar la arquitectura de datos, los desarrolladores deben elegir entre modelos relacionales y NoSQL según las necesidades del proyecto.
Las bases de datos relacionales, como MySQL o SQL Server, utilizan tablas estructuradas y son ideales para mantener la integridad en transacciones complejas.
Por otro lado, las soluciones NoSQL, como Firebase Firestore, guardan la información en colecciones de documentos flexibles.
Estos esquemas dinámicos son perfectos para aplicaciones que requieren sincronización de datos en tiempo real o donde la estructura de los datos cambia constantemente.
Una mala elección en el tipo de base de datos puede generar cuellos de botella severos al escalar la aplicación.
----------------------